In [1]:
!nvidia-smi

Thu Feb 26 19:50:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install libraries
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate peft evaluate
!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import time
import re
import torch
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

# Load Financial PhraseBank (all-agree subset for clean labels)
print("Loading Financial PhraseBank...")

# After (proper parquet, no config name needed)
ds_full = load_dataset("FinanceMTEB/financial_phrasebank")
ds = ds_full["test"]   # 1,000 rows held-out test split

# Labels: 0 = negative, 1 = neutral, 2 = positive
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
INT_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

print(f"Dataset size: {len(ds)} examples")
print(f"Label distribution: {dict(zip(*np.unique(ds['label'], return_counts=True)))}")

Loading Financial PhraseBank...


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/80.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1264 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset size: 1000 examples
Label distribution: {np.int64(0): np.int64(138), np.int64(1): np.int64(612), np.int64(2): np.int64(250)}


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Config ---
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
BATCH_SIZE = 4           # 4B model with 4-bit is memory efficient
MAX_NEW_TOKENS = 64      # Short output needed for sentiment label
EVAL_SIZE = len(ds)      # Run full dataset

# --- Load Model with 4-bit Quantization ---
print("Loading Llama-3-8B with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map="auto",
)
model.eval()

print("Model loaded successfully!")


Loading Llama-3-8B with 4-bit quantization...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/Llama3_8B_FinancialPhraseBank_Eval"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print(f"Saving checkpoints to: {DRIVE_SAVE_DIR}")


Mounted at /content/drive
Saving checkpoints to: /content/drive/MyDrive/Colab_Data/Llama3_8B_FinancialPhraseBank_Eval


In [10]:
import os
import json

CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_fpb_eval_checkpoint.jsonl")
results = []

# --- Resume from checkpoint if available ---
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint. Loading previous results...")
    with open(CHECKPOINT_FILE, "r") as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Resuming from example {len(results)}.")
else:
    print("No checkpoint found. Starting from scratch.")

# --- Prompt & Label Extraction Helpers ---
SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment of the following financial news sentence as exactly one of: "
    "positive, negative, or neutral. "
    "Respond with only the single word label."
)

def build_prompt(sentence: str) -> str:
    """Build a chat-formatted prompt using Llama 3 template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sentence},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def extract_label(text: str):
    """Parse model output to one of positive / neutral / negative."""
    text = text.strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in text:
            return label
    return None  # unparseable

# --- Prepare Data ---
sentences   = ds["text"]
int_labels  = ds["label"]  # 0=negative, 1=neutral, 2=positive
str_labels  = [LABEL_MAP[l] for l in int_labels]

start_idx = len(results)

print(f"Starting batched evaluation (Batch Size: {BATCH_SIZE})...")

# --- Batched Evaluation Loop ---
for i in tqdm(range(start_idx, EVAL_SIZE, BATCH_SIZE)):
    batch_sentences  = sentences[i : i + BATCH_SIZE]
    batch_gt         = str_labels[i : i + BATCH_SIZE]

    batch_texts = [build_prompt(s) for s in batch_sentences]
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    batch_start = time.time()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    batch_time = time.time() - batch_start

    new_tokens    = generated_ids[:, inputs.input_ids.shape[1]:]
    decoded       = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    batch_results = []
    for j, response in enumerate(decoded):
        pred  = extract_label(response)
        gt    = batch_gt[j]
        is_correct  = (pred == gt)
        is_unknown  = (pred is None)

        res_item = {
            "idx":             i + j,
            "sentence":        batch_sentences[j],
            "ground_truth":    gt,
            "prediction":      pred,
            "raw_response":    response.strip(),
            "is_correct":      is_correct,
            "is_unknown":      is_unknown,
            "new_tokens":      int(new_tokens[j].numel()),
            "batch_time_share": batch_time / len(batch_sentences),
        }
        batch_results.append(res_item)
        results.append(res_item)

    # Checkpoint after every batch
    with open(CHECKPOINT_FILE, "a") as f:
        for item in batch_results:
            f.write(json.dumps(item) + "")

No checkpoint found. Starting from scratch.
Starting batched evaluation (Batch Size: 4)...


100%|██████████| 250/250 [02:44<00:00,  1.52it/s]


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

# --- Final Metrics ---
all_gt   = [r["ground_truth"] for r in results]
all_pred = [r["prediction"] if r["prediction"] is not None else "unknown" for r in results]

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
total_new_tokens   = sum(r["new_tokens"] for r in results)
total_gen_time_sum = sum(r["batch_time_share"] for r in results)

final_metrics = {
    "method":              "0_shot",
    "model":               MODEL_NAME,
    "dataset":             "financial_phrasebank/sentences_allagree",
    "eval_size":           len(results),
    "accuracy":            correct_count / len(results) if results else 0,
    "unknown_frac":        unknown_count / len(results) if results else 0,
    "gen_tokens_per_sec":  total_new_tokens / total_gen_time_sum if total_gen_time_sum > 0 else 0,
    "total_gen_time_min":  total_gen_time_sum / 60,
}

print("=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"{k}: {v}")

print("=== Per-Class Report ===")
labels = ["negative", "neutral", "positive"]
print(classification_report(all_gt, all_pred, labels=labels))

print("=== Confusion Matrix (rows=true, cols=pred) ===")
print(labels)
print(confusion_matrix(all_gt, all_pred, labels=labels))

# Save final metrics to Drive
metrics_path = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_fpb_final_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")


=== Final Metrics ===
method: 0_shot
model: meta-llama/Meta-Llama-3-8B-Instruct
dataset: financial_phrasebank/sentences_allagree
eval_size: 1000
accuracy: 0.906
unknown_frac: 0.0
gen_tokens_per_sec: 12.338917767213431
total_gen_time_min: 2.701479494571686
=== Per-Class Report ===
              precision    recall  f1-score   support

    negative       0.93      0.91      0.92       138
     neutral       0.88      0.99      0.93       612
    positive       0.97      0.71      0.82       250

    accuracy                           0.91      1000
   macro avg       0.93      0.87      0.89      1000
weighted avg       0.91      0.91      0.90      1000

=== Confusion Matrix (rows=true, cols=pred) ===
['negative', 'neutral', 'positive']
[[126  12   0]
 [  4 603   5]
 [  5  68 177]]
Metrics saved to /content/drive/MyDrive/Colab_Data/Llama3_8B_FinancialPhraseBank_Eval/llama3_8b_fpb_final_metrics.json
